# gpu-benches-baseliner - figure regeneration
Run top to bottom from a clone. Reads `logs/`, writes `figures/`. Backend and card are read from each `logs/<campaign>/metadata.json`; no local paths.

In [ ]:
# gpu-benches-baseliner - figure regeneration
# Regenerates every figure in RESULTS.md from the campaigns kept under logs/.
# Reads logs/<campaign>/metadata.json to detect backend and card; no local paths.
import json, re
from pathlib import Path
import matplotlib, matplotlib.pyplot as plt, matplotlib.ticker, matplotlib.colors
import numpy as np, pandas as pd

REPO = Path.cwd()
LOGS = REPO / "logs"
FIGDIR = REPO / "figures"
plt.rcParams["figure.dpi"] = 110

# Per-card hardware constants used by the secondary axes and the roofline / peak-% readings.
HW = {
    "NVIDIA GeForce RTX 2080 Ti": dict(dram_gbs=616, pcie_gbs=15.75, fp32_tflops=13.45,
                                       units=68, clock_ghz=1.545),
    "AMD Instinct MI210":         dict(dram_gbs=1638, pcie_gbs=31.5, fp32_tflops=22.6,
                                       units=104, clock_ghz=1.7),
}
# How each campaign maps to a report part: short label, colour, output dir, filename prefix.
LAYOUT = {
    "Log_Test_RTX2080ti_cuda": dict(short="CUDA / 2080 Ti", colour="#0072B2",
                                    subdir="part1_cuda_2080ti", prefix="p1_cuda2080"),
    "Log_Test_RTX2080ti_hip":  dict(short="HIP / 2080 Ti", colour="#D55E00",
                                    subdir="part2_cuda_vs_hip_2080ti", prefix="p2_cuda_vs_hip"),
    "Log_Test_MI210":          dict(short="HIP / MI210", colour="#009E73",
                                    subdir="part3_hip_mi210", prefix="p3_mi210"),
}

def _discover():
    camps = {}
    for meta in sorted(LOGS.glob("*/metadata.json")):
        m = json.loads(meta.read_text())
        name = meta.parent.name
        lay = LAYOUT.get(name, dict(short=name, colour="#0072B2", subdir=name, prefix=name))
        card = next((c for c in HW if c.split()[-1] in m["gpu"] or c in m["gpu"]), None)
        camps[name] = dict(dir=meta.parent, backend=m["backend"], card=card,
                           gpu=m["gpu"], **lay, **HW.get(card, {}))
    return camps

CAMPAIGNS = _discover()
# Stable handles for the three report parts.
C_CUDA = "Log_Test_RTX2080ti_cuda"
C_HIP  = "Log_Test_RTX2080ti_hip"
C_MI   = "Log_Test_MI210"

BENCHES = ["gpu-latency", "gpu-cache", "gpu-l2-stream", "gpu-strides", "gpu-small-kernels",
           "gpu-roofline", "gpu-memcpy", "gpu-umstream", "gpu-incore"]
C_REF = "0.35"
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]
MARKERS = ["o", "s", "^", "d", "v", "P"]
KER_INCORE = ["fma-mixed", "fma-separated", "div", "sqrt"]

def run_colors(n): return [plt.cm.viridis(x) for x in np.linspace(0.05, 0.92, n)]
def serie_colors(n): return PAL[:n] if n <= len(PAL) else [plt.cm.tab10(i % 10) for i in range(n)]

def _num(s):
    try:
        f = float(s); return int(f) if f.is_integer() else f
    except (TypeError, ValueError): return s

def runs_of(camp, bench):
    d = CAMPAIGNS[camp]["dir"]
    ns = [int(mt.group(1)) for f in d.glob(f"{bench}.run*.json")
          for mt in [re.fullmatch(rf"{re.escape(bench)}\.run(\d+)\.json", f.name)] if mt]
    return sorted(ns)

def load_run(camp, bench, run):
    c = CAMPAIGNS[camp]
    # try padded then unpadded
    p = c["dir"] / f"{bench}.run{run:02d}.json"
    if not p.exists(): p = c["dir"] / f"{bench}.run{run}.json"
    br = json.loads(p.read_bytes())["campaign_runs"][0]["benchmark_runs"]
    results = br[c["backend"]][bench]["benchmark_report"]["results"]
    rows = []
    for r in results:
        row = {}
        for opts in r["sweep_point"].values():
            for k, v in opts.items(): row[k] = _num(v["value"])
        for m in r["measurements"]:
            if not isinstance(m["data"], list): row[m["name"]] = m["data"]
        rows.append(row)
    df = pd.DataFrame(rows); df.insert(0, "run", run); return df

_CACHE = {}
def load(camp, bench):
    if (camp, bench) in _CACHE: return _CACHE[(camp, bench)]
    ns = runs_of(camp, bench)
    if not ns: raise FileNotFoundError(f"{camp}/{bench}: no result files")
    df = pd.concat([load_run(camp, bench, n) for n in ns], ignore_index=True)
    if bench == "gpu-l2-stream":
        df["footprint_kb"] = np.floor(df.length * 16 / 1024)
    if bench == "gpu-small-kernels":
        df["volume_kb"] = df["size"] * 16 / 1024
        df["t_s"] = df["median"] * 1e-3
        df["volume_o"] = df["size"] * 16
    if bench in SERIE_COMPOSEE:
        cols = SERIE_COMPOSEE[bench]
        df["serie"] = df[cols[0]].astype(str)
        for c in cols[1:]: df["serie"] = df["serie"] + " / " + df[c].astype(str)
    _CACHE[(camp, bench)] = df; return df

def fmt_kb(x, pos=None):
    if x < 1024: return f"{x:.3g} kB"
    if x < 1024 * 1024: return f"{x/1024:.3g} MB"
    return f"{x/1024/1024:.3g} GB"

def out(camp, bench, part_prefix=None):
    c = CAMPAIGNS[camp]
    pref = part_prefix or c["prefix"]
    sub = FIGDIR / c["subdir"]; sub.mkdir(parents=True, exist_ok=True)
    return sub / f"{pref}_{bench.replace('-', '_')}.png"

RUNS = {c: runs_of(c, "gpu-incore") for c in CAMPAIGNS}
COLORS = {c: dict(zip(RUNS[c], run_colors(len(RUNS[c])))) for c in CAMPAIGNS}

SPECS = {
    "gpu-latency": dict(x="buffer_size_kb", y="latency_ns", series=None, xscale="log",
        yscale="linear", xmul_kb=1, bpc=None, cycles=True, better="min",
        xlabel="Data volume (buffer size)", ylabel="Latency (ns)",
        titre="pointer chasing: latency vs footprint"),
    "gpu-cache": dict(x="working_set_kb", y="memory_bandwidth", series=None, xscale="log",
        yscale="linear", xmul_kb=1, bpc="device", better="max",
        xlabel="Data volume per unit (working set)", ylabel="Bandwidth (GB/s)",
        titre="bandwidth of the first two cache levels"),
    "gpu-l2-stream": dict(x="footprint_kb", y="memory_bandwidth", series="kernel_type",
        xscale="log", yscale="linear", xmul_kb=1, bpc="device", better="max",
        xlabel="Footprint (length x 16 bytes)", ylabel="Bandwidth (GB/s)",
        titre="shared cache bandwidth"),
    "gpu-strides": dict(x="arg", y="memory_bandwidth", series="serie", xscale="linear",
        yscale="linear", xmul_kb=None, bpc="one", better="max",
        xlabel="Access stride (arg, in elements)", ylabel="Bandwidth (GB/s, 1 unit)",
        titre="coalescing and cache banks"),
    "gpu-small-kernels": dict(x="volume_kb", y="memory_bandwidth", series="block_size",
        xscale="log", yscale="linear", xmul_kb=1, bpc="device", better="max",
        xlabel="Data volume moved (16 bytes per element)", ylabel="Bandwidth (GB/s)",
        titre="launch overhead versus bandwidth"),
    "gpu-roofline": dict(x="arithmetic_intensity", y="arithmetic_bandwidth", series=None,
        xscale="log", yscale="log", xmul_kb=None, bpc=None, better="max",
        xlabel="Arithmetic intensity (FLOP / byte)", ylabel="Throughput (GFLOP/s)",
        titre="roofline: compute throughput vs arithmetic intensity"),
    "gpu-memcpy": dict(x="transfer_kb", y="memory_bandwidth", series=None, xscale="log",
        yscale="linear", xmul_kb=1, bpc=None, better="max",
        xlabel="Transfer size", ylabel="Bandwidth (GB/s)", titre="host <-> device transfers"),
    "gpu-umstream": dict(x="transfer_mb", y="memory_bandwidth", series=None, xscale="log",
        yscale="log", xmul_kb=1024, bpc=None, better="max",
        xlabel="transfer_mb (per array)", ylabel="Bandwidth (GB/s)",
        titre="prefetched unified memory"),
    "gpu-incore": dict(x="warp_count", y="rcp_throughput", series="serie", fix={"ilp": 1},
        xscale="log", yscale="log", xmul_kb=None, bpc=None, better="min",
        xlabel="warp_count (block_size = 32 x warp_count)",
        ylabel="rcp_throughput (cycles/op)", titre="cycles per operation vs TLP, at ILP = 1"),
}
SERIE_COMPOSEE = {"gpu-strides": ["kernel_type", "precision"],
                  "gpu-incore": ["precision", "kernel_type"]}
ORDRES = {"kernel_type": ["read", "write", "scale", "triad", "stride", "block",
                          "fma-mixed", "fma-separated", "div", "sqrt"],
          "precision": ["float", "double"]}
SORTIE_DEDIEE = {"gpu-incore", "gpu-strides", "gpu-small-kernels"}

def apply_fix(df, spec):
    for k, v in spec.get("fix", {}).items(): df = df[df[k] == v]
    return df

def serie_values(df, spec):
    if not spec["series"]: return [None]
    u = list(df[spec["series"]].unique())
    ordre = ORDRES.get(spec["series"])
    if ordre: return [v for v in ordre if v in u] + [v for v in u if v not in ordre]
    if spec["series"] == "serie":
        return sorted(u, key=lambda s: [ORDRES["kernel_type"].index(p)
                      if p in ORDRES["kernel_type"] else 99 for p in str(s).split(" / ")])
    return sorted(u)

def deco_axes(ax, spec, camp=None, secondaire=True):
    if spec["xscale"] == "log": ax.set_xscale("log", base=2)
    if spec["yscale"] == "log": ax.set_yscale("log")
    else: ax.set_ylim(bottom=0)
    if spec["xmul_kb"]:
        mul = spec["xmul_kb"]
        ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, p: fmt_kb(v * mul)))
    if spec["xscale"] == "log":
        ax.xaxis.set_major_locator(matplotlib.ticker.LogLocator(base=2, subs=(1.0,), numticks=8))
        if not spec["xmul_kb"]:
            ax.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(True, which="both", alpha=0.25)
    if camp and secondaire and spec["yscale"] == "linear":
        c = CAMPAIGNS[camp]
        if spec.get("bpc") and c.get("clock_ghz"):
            n = c["units"] if spec["bpc"] == "device" else 1
            k = c["clock_ghz"] * n
            sec = ax.secondary_yaxis("right", functions=(lambda y: y / k, lambda y: y * k))
            sec.set_ylabel(f"bytes / cycle / {'unit' if n == 1 else 'SM-CU'} (at {c['clock_ghz']} GHz)", fontsize=9)
        elif spec.get("cycles") and c.get("clock_ghz"):
            k = c["clock_ghz"]
            sec = ax.secondary_yaxis("right", functions=(lambda ns: ns * k, lambda cy: cy / k))
            sec.set_ylabel(f"Latency (cycles at {c['clock_ghz']} GHz)", fontsize=9)
    return ax

def spread(df, spec):
    cles = ([spec["series"]] if spec["series"] else []) + [spec["x"]]
    g = df.groupby(cles)[spec["y"]]
    o = g.agg(med="median", moy="mean", lo="min", hi="max", ec="std", n="size").reset_index()
    o["cov_pct"] = 100 * o.ec / o.moy.abs(); return o

# ---- curve figure (parts 1 & 3) ----------------------------------------------------------
DISP = {}   # (camp, bench) -> spread, feeds the reproducibility synthesis
def figure_runs(camp, bench, figsize=(13, 6)):
    spec = SPECS[bench]; df = apply_fix(load(camp, bench), spec)
    vals = serie_values(df, spec)
    fig, ax = plt.subplots(figsize=figsize)
    if spec["series"]:
        cols = serie_colors(len(vals))
        for v, c, mk in zip(vals, cols, MARKERS * 3):
            d0 = df[df[spec["series"]] == v]
            for r in RUNS[camp]:
                d = d0[d0.run == r].sort_values(spec["x"])
                ax.plot(d[spec["x"]], d[spec["y"]], color=c, linewidth=0.7, alpha=0.45, zorder=2)
            m = d0.groupby(spec["x"])[spec["y"]].median().sort_index()
            ax.plot(m.index, m.values, color=c, linewidth=1.9, marker=mk, markersize=3.4,
                    zorder=3, label=str(v))
        leg = {"kernel_type": "kernel", "block_size": "block size",
               "serie": "kernel / precision"}.get(spec["series"], spec["series"])
        ax.legend(fontsize=8.5, ncol=2 if len(vals) > 4 else 1, title=leg, title_fontsize=9)
    else:
        for r in RUNS[camp]:
            d = df[df.run == r].sort_values(spec["x"])
            ax.plot(d[spec["x"]], d[spec["y"]], marker="o", markersize=2.6, linewidth=1.1,
                    alpha=0.9, color=COLORS[camp][r], label=f"run {r}")
        ax.legend(fontsize=7, ncol=2, loc="best")
    deco_axes(ax, spec, camp)
    ax.set_xlabel(spec["xlabel"]); ax.set_ylabel(spec["ylabel"])
    ax.set_title(f"{CAMPAIGNS[camp]['short']} - {bench}: {spec['titre']}", fontsize=12.5, fontweight="bold")
    fig.tight_layout(); fig.savefig(out(camp, bench), dpi=150, bbox_inches="tight"); plt.show()
    DISP[(camp, bench)] = spread(df, spec)

# ---- dedicated layouts -------------------------------------------------------------------
def fit_overhead_bw(vol_o, t_s):
    pente, ordonnee = np.polyfit(np.asarray(vol_o), np.asarray(t_s), 1)
    return ordonnee, 1 / pente

def smallkernels_fits(camp):
    df = load(camp, "gpu-small-kernels"); o = {}
    for bs in sorted(df.block_size.unique()):
        m = df[df.block_size == bs].groupby("volume_o").t_s.median().sort_index()
        o[bs] = fit_overhead_bw(m.index.values, m.values)
    return o

def _incore_grids(camp):
    inc = load(camp, "gpu-incore"); ilps, wc = sorted(inc.ilp.unique()), sorted(inc.warp_count.unique())
    o = {}
    for prec in ["float", "double"]:
        for k in KER_INCORE:
            d = inc[(inc.precision == prec) & (inc.kernel_type == k)]
            med = d.pivot_table(index="warp_count", columns="ilp", values="rcp_throughput",
                                aggfunc="median").reindex(index=wc, columns=ilps)
            cov = (d.groupby(["warp_count", "ilp"]).rcp_throughput
                   .agg(lambda s: 100 * s.std() / s.mean()).unstack().reindex(index=wc, columns=ilps))
            o[(prec, k)] = (med, cov)
    return o, ilps, wc

def figure_incore_table(camp):
    grids, ilps, wc = _incore_grids(camp)
    fig, axes = plt.subplots(2, len(KER_INCORE), figsize=(4.15 * len(KER_INCORE), 7.2), squeeze=False)
    for row, prec in enumerate(["float", "double"]):
        for col, k in enumerate(KER_INCORE):
            ax = axes[row, col]; med, cov = grids[(prec, k)]
            ax.imshow(np.log10(med.values), cmap="RdYlGn_r", aspect="auto")
            for i in range(med.shape[0]):
                for j in range(med.shape[1]):
                    ax.text(j, i - 0.12, f"{med.values[i, j]:.3g}", ha="center", va="center",
                            fontsize=7.5, color="0.1")
                    if cov.values[i, j] > 1:
                        ax.text(j, i + 0.26, f"±{cov.values[i, j]:.0f}%", ha="center",
                                va="center", fontsize=5.8, color="0.3", style="italic")
            ax.set_xticks(range(len(ilps)), [str(x) for x in ilps])
            ax.set_yticks(range(len(wc)), [str(x) for x in wc])
            if row == 0: ax.set_title(k, fontweight="bold", fontsize=11)
            if row == 1: ax.set_xlabel("ILP")
            if col == 0: ax.set_ylabel(f"{prec}\nwarp_count (TLP)", fontsize=10)
    fig.suptitle(f"{CAMPAIGNS[camp]['short']} - gpu-incore: ILP x TLP table (cycles/op)",
                 fontsize=12.5, fontweight="bold")
    fig.tight_layout(); fig.savefig(out(camp, "gpu-incore"), dpi=150, bbox_inches="tight"); plt.show()
    DISP[(camp, "gpu-incore")] = spread(apply_fix(load(camp, "gpu-incore"), SPECS["gpu-incore"]), SPECS["gpu-incore"])

def figure_strides_table(camp):
    c = CAMPAIGNS[camp]; st = load(camp, "gpu-strides")
    st = st[(st.kernel_type == "stride") & (st.arg >= 1)]
    st = st.assign(bpc=st.memory_bandwidth / c["clock_ghz"])
    NCOL = 16
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.4))
    for ax, prec in zip(axes, ["double", "float"]):
        d = st[st.precision == prec].groupby("arg").bpc.median().sort_index()
        nrow = int(np.ceil(len(d) / NCOL)); grid = np.full(nrow * NCOL, np.nan)
        grid[:len(d)] = d.values; grid = grid.reshape(nrow, NCOL)
        ax.imshow(grid, cmap="RdYlGn", aspect="auto", vmin=0)
        for i in range(nrow):
            for j in range(NCOL):
                if not np.isnan(grid[i, j]):
                    ax.text(j, i, f"{np.ceil(grid[i, j]):.0f}", ha="center", va="center",
                            fontsize=9, fontweight="bold", color="0.1")
        ax.set_xticks(range(NCOL), [str(j + 1) for j in range(NCOL)], fontweight="bold", fontsize=9)
        ax.xaxis.set_ticks_position("top")
        ax.set_yticks(range(nrow), [str(i * NCOL + 1) for i in range(nrow)], fontweight="bold", fontsize=9)
        ax.tick_params(length=0)
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_ylabel("Stride", fontweight="bold", fontsize=10, rotation=0, ha="right", va="center", labelpad=14)
        ax.set_title(f"{c['short']} - L1 cache: bytes/cycle ({prec})", fontweight="bold", fontsize=11, pad=20)
    fig.tight_layout(h_pad=2.2); fig.savefig(out(camp, "gpu-strides"), dpi=150, bbox_inches="tight"); plt.show()
    DISP[(camp, "gpu-strides")] = spread(apply_fix(load(camp, "gpu-strides"), SPECS["gpu-strides"]), SPECS["gpu-strides"])

def figure_smallkernels_fit(camp):
    df = load(camp, "gpu-small-kernels"); BS = sorted(df.block_size.unique())
    cols = dict(zip(BS, serie_colors(len(BS)))); fits = smallkernels_fits(camp)
    fig, axd = plt.subplot_mosaic([["fit", "a"], ["fit", "b"]], figsize=(15, 6.4), width_ratios=[2, 1])
    ax = axd["fit"]
    for bs in BS:
        m = df[df.block_size == bs].groupby("volume_o").t_s.median().sort_index()
        ax.plot(m.index / 1024, m.values * 1e6, "o", markersize=2.4, alpha=0.6, color=cols[bs], label=f"{bs}")
        a, b = fits[bs]; v = np.geomspace(m.index.min(), m.index.max(), 200)
        ax.plot(v / 1024, (a + v / b) * 1e6, color=cols[bs], linewidth=1.5)
    ax.set_xscale("log", base=2); ax.set_yscale("log")
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, p: fmt_kb(v)))
    ax.xaxis.set_major_locator(matplotlib.ticker.LogLocator(base=2, subs=(1.0,), numticks=7))
    ax.set_xlabel("Data volume moved   -   points: measurement, line: model")
    ax.set_ylabel("Median time (microseconds)"); ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8, title="block_size", title_fontsize=9, ncol=2)
    x = np.arange(len(BS))
    axd["a"].bar(x, [fits[bs][0] * 1e6 for bs in BS], color=[cols[bs] for bs in BS])
    axd["a"].set_ylabel("a: launch\noverhead (us)", fontsize=9)
    axd["b"].bar(x, [fits[bs][1] / 1e9 for bs in BS], color=[cols[bs] for bs in BS])
    axd["b"].set_ylabel("b: bandwidth\n(GB/s)", fontsize=9)
    for k in ("a", "b"):
        axd[k].set_xticks(x, [str(v) for v in BS], fontsize=8); axd[k].grid(True, axis="y", alpha=0.25)
    axd["b"].set_xlabel("block_size", fontsize=9)
    fig.suptitle(f"{CAMPAIGNS[camp]['short']} - gpu-small-kernels: model fit T = a + V/b",
                 fontsize=12.5, fontweight="bold")
    fig.tight_layout(); fig.savefig(out(camp, "gpu-small-kernels"), dpi=150, bbox_inches="tight"); plt.show()
    DISP[(camp, "gpu-small-kernels")] = spread(apply_fix(load(camp, "gpu-small-kernels"), SPECS["gpu-small-kernels"]), SPECS["gpu-small-kernels"])

def figure_benchmark(camp, bench):
    if bench == "gpu-incore": figure_incore_table(camp)
    elif bench == "gpu-strides": figure_strides_table(camp)
    elif bench == "gpu-small-kernels": figure_smallkernels_fit(camp)
    else: figure_runs(camp, bench)

# ---- part 2: CUDA vs HIP on the same card ------------------------------------------------
CMP = [C_CUDA, C_HIP]
CMP_RES = {}   # bench -> dict(rap, better, chev), feeds the part-2 synthesis

def _cmp_out(bench):
    sub = FIGDIR / CAMPAIGNS[C_HIP]["subdir"]; sub.mkdir(parents=True, exist_ok=True)
    return sub / f"p2_cuda_vs_hip_{bench.replace('-', '_')}.png"

def _store_compare(bench):
    spec = SPECS[bench]; cles = ([spec["series"]] if spec["series"] else []) + [spec["x"]]
    moys = {c: (apply_fix(load(c, bench), spec).groupby(cles)[spec["y"]]
               .agg(moy="mean", lo="min", hi="max").reset_index().set_index(cles)) for c in CMP}
    a, b = moys[CMP[0]], moys[CMP[1]]; idx = a.index.intersection(b.index)
    rap = (b.loc[idx, "moy"] / a.loc[idx, "moy"]).replace([np.inf, -np.inf], np.nan).dropna()
    chev = ((a.loc[idx, "hi"] >= b.loc[idx, "lo"]) & (b.loc[idx, "hi"] >= a.loc[idx, "lo"]))
    CMP_RES[bench] = dict(rap=rap, better=spec["better"], chev=float(chev.mean()))

def figure_compare(bench):
    spec = SPECS[bench]; cles = ([spec["series"]] if spec["series"] else []) + [spec["x"]]
    moys = {c: apply_fix(load(c, bench), spec).groupby(cles)[spec["y"]]
            .agg(moy="mean", lo="min", hi="max").reset_index() for c in CMP}
    vals = serie_values(apply_fix(load(CMP[0], bench), spec), spec)
    cols = dict(zip(vals, serie_colors(len(vals))))
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(13, 7.6), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1.25], "hspace": 0.08})
    for v in vals:
        for camp, ls in zip(CMP, ["-", "--"]):
            d = moys[camp]
            if spec["series"]: d = d[d[spec["series"]] == v]
            d = d.sort_values(spec["x"])
            c = cols[v] if spec["series"] else CAMPAIGNS[camp]["colour"]
            ax.fill_between(d[spec["x"]], d.lo, d.hi, color=c, alpha=0.18, linewidth=0)
            lab = (CAMPAIGNS[camp]["short"] if not spec["series"]
                   else (f"{v}" if camp == CMP[0] else None))
            ax.plot(d[spec["x"]], d.moy, ls, color=c, linewidth=1.6, label=lab)
    if spec["series"]:
        h_style = [matplotlib.lines.Line2D([], [], color="0.3", linestyle=ls,
                   label=CAMPAIGNS[camp]["short"]) for camp, ls in zip(CMP, ["-", "--"])]
        leg1 = ax.legend(fontsize=8.5, ncol=2 if len(vals) > 4 else 1, loc="lower right")
        ax.add_artist(leg1); ax.legend(handles=h_style, fontsize=8.5, loc="upper left")
    else:
        ax.legend(fontsize=9, loc="best")
    deco_axes(ax, spec, CMP[0]); ax.set_ylabel(spec["ylabel"])
    ax.set_title(f"RTX 2080 Ti - {bench}: {spec['titre']}", fontsize=12.5, fontweight="bold")
    for v in vals:
        a_ = moys[CMP[0]]; b_ = moys[CMP[1]]
        if spec["series"]: a_, b_ = a_[a_[spec["series"]] == v], b_[b_[spec["series"]] == v]
        a_ = a_.set_index(spec["x"]).moy; b_ = b_.set_index(spec["x"]).moy
        idx = a_.index.intersection(b_.index)
        axr.plot(idx, 100 * (b_.loc[idx] / a_.loc[idx] - 1),
                 color=cols[v] if spec["series"] else "0.25", linewidth=1.3)
    axr.axhline(0, color="0.2", linestyle="--", linewidth=1.2)
    axr.set_ylabel("HIP vs CUDA\n(%)", fontsize=9); axr.set_xlabel(spec["xlabel"])
    axr.grid(True, which="both", alpha=0.25)
    axr.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, p: f"{v:+g}"))
    fig.savefig(_cmp_out(bench), dpi=150, bbox_inches="tight"); plt.show()
    _store_compare(bench)

def figure_incore_table_ratio():
    moys = {}
    for camp in CMP:
        inc = load(camp, "gpu-incore")
        moys[camp] = inc.groupby(["precision", "kernel_type", "warp_count", "ilp"]).rcp_throughput.mean()
    idx = moys[CMP[0]].index.intersection(moys[CMP[1]].index)
    rap = (moys[CMP[1]].loc[idx] / moys[CMP[0]].loc[idx]).rename("rapport").reset_index()
    ilps = sorted(rap.ilp.unique()); wc = sorted(rap.warp_count.unique())
    dev = float(np.nanmax(np.abs(rap.rapport - 1))) or 1e-3
    norm = matplotlib.colors.Normalize(vmin=1 - dev, vmax=1 + dev)
    fig, axes = plt.subplots(2, len(KER_INCORE), figsize=(4.15 * len(KER_INCORE), 7.2), squeeze=False)
    im = None
    for row, prec in enumerate(["float", "double"]):
        for col, k in enumerate(KER_INCORE):
            ax = axes[row, col]
            m = (rap[(rap.precision == prec) & (rap.kernel_type == k)]
                 .pivot_table(index="warp_count", columns="ilp", values="rapport")
                 .reindex(index=wc, columns=ilps))
            im = ax.imshow(m.values, cmap="RdBu_r", norm=norm, aspect="auto")
            for i in range(m.shape[0]):
                for j in range(m.shape[1]):
                    v = m.values[i, j]
                    if not np.isfinite(v): continue
                    pct = 100 * (v - 1)
                    txt = f"{pct:+.1f} %" if abs(pct) >= 0.05 else "~0 %"
                    ax.text(j, i, txt, ha="center", va="center", fontsize=7.5, color="0.1")
            ax.set_xticks(range(len(ilps)), [str(x) for x in ilps])
            ax.set_yticks(range(len(wc)), [str(x) for x in wc])
            if row == 0: ax.set_title(k, fontweight="bold", fontsize=11)
            if row == 1: ax.set_xlabel("ILP")
            if col == 0: ax.set_ylabel(f"{prec}\nwarp_count (TLP)", fontsize=10)
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.015)
    cb.set_label("HIP / CUDA ratio  (red = HIP slower)", fontsize=9)
    fig.suptitle("RTX 2080 Ti - gpu-incore: HIP vs CUDA, ILP x TLP table", fontsize=12.5, fontweight="bold")
    fig.savefig(_cmp_out("gpu-incore"), dpi=150, bbox_inches="tight"); plt.show()
    _store_compare("gpu-incore")

def figure_strides_table_ratio():
    moys = {}
    for camp in CMP:
        st = load(camp, "gpu-strides"); st = st[(st.kernel_type == "stride") & (st.arg >= 1)]
        moys[camp] = st.groupby(["precision", "arg"]).memory_bandwidth.mean()
    idx = moys[CMP[0]].index.intersection(moys[CMP[1]].index)
    rap = (moys[CMP[1]].loc[idx] / moys[CMP[0]].loc[idx])
    NCOL = 16; dev = float(np.nanmax(np.abs(rap - 1))) or 1e-3
    norm = matplotlib.colors.Normalize(vmin=1 - dev, vmax=1 + dev)
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.4)); im = None
    for ax, prec in zip(axes, ["double", "float"]):
        d = rap.loc[prec].sort_index()
        nrow = int(np.ceil(len(d) / NCOL)); grid = np.full(nrow * NCOL, np.nan)
        grid[:len(d)] = d.values; grid = grid.reshape(nrow, NCOL)
        im = ax.imshow(grid, cmap="RdBu_r", norm=norm, aspect="auto")
        for i in range(nrow):
            for j in range(NCOL):
                if not np.isnan(grid[i, j]):
                    pct = 100 * (grid[i, j] - 1)
                    ax.text(j, i, f"{pct:+.1f}" if abs(pct) >= 0.05 else "~0", ha="center",
                            va="center", fontsize=8.5, color="0.1")
        ax.set_xticks(range(NCOL), [str(j + 1) for j in range(NCOL)], fontweight="bold", fontsize=9)
        ax.xaxis.set_ticks_position("top")
        ax.set_yticks(range(nrow), [str(i * NCOL + 1) for i in range(nrow)], fontweight="bold", fontsize=9)
        ax.tick_params(length=0)
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_ylabel("Stride", fontweight="bold", fontsize=10, rotation=0, ha="right", va="center", labelpad=14)
        ax.set_title(f"RTX 2080 Ti - gpu-strides: HIP vs CUDA, % ({prec})", fontweight="bold", fontsize=11, pad=20)
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.015)
    cb.set_label("HIP / CUDA ratio  (red = HIP slower)", fontsize=9)
    fig.savefig(_cmp_out("gpu-strides"), dpi=150, bbox_inches="tight"); plt.show()
    _store_compare("gpu-strides")

def figure_smallkernels_fit_compare():
    fits = {camp: smallkernels_fits(camp) for camp in CMP}; BS = sorted(fits[CMP[0]])
    x = np.arange(len(BS)); w = 0.38
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.8))
    for i, camp in enumerate(CMP):
        c = CAMPAIGNS[camp]["colour"]
        axa.bar(x + (i - 0.5) * w, [fits[camp][bs][0] * 1e6 for bs in BS], w, color=c, label=CAMPAIGNS[camp]["short"])
        axb.bar(x + (i - 0.5) * w, [fits[camp][bs][1] / 1e9 for bs in BS], w, color=c, label=CAMPAIGNS[camp]["short"])
    axa.set_ylabel("a: launch overhead (microseconds)"); axa.set_title("Launch overhead", fontweight="bold")
    axb.set_ylabel("b: asymptotic bandwidth (GB/s)"); axb.set_title("Asymptotic bandwidth", fontweight="bold")
    for a_ in (axa, axb):
        a_.set_xticks(x, [str(v) for v in BS], fontsize=8.5); a_.set_xlabel("block_size"); a_.grid(True, axis="y", alpha=0.25)
    axa.legend(fontsize=8.5)
    fig.suptitle("RTX 2080 Ti - gpu-small-kernels: model parameters per backend", fontsize=12.5, fontweight="bold")
    fig.tight_layout(); fig.savefig(_cmp_out("gpu-small-kernels"), dpi=150, bbox_inches="tight"); plt.show()
    _store_compare("gpu-small-kernels")

def figure_benchmark_compare(bench):
    if bench == "gpu-incore": figure_incore_table_ratio()
    elif bench == "gpu-strides": figure_strides_table_ratio()
    elif bench == "gpu-small-kernels": figure_smallkernels_fit_compare()
    else: figure_compare(bench)

# ---- synthesis figures -------------------------------------------------------------------
def figure_repro_part1():
    order = sorted([b for b in BENCHES if (C_CUDA, b) in DISP],
                   key=lambda b: DISP[(C_CUDA, b)].cov_pct.median())
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14.5, 5), gridspec_kw={"width_ratios": [1.25, 1]})
    data = [DISP[(C_CUDA, b)].cov_pct.replace([np.inf, -np.inf], np.nan).dropna().values for b in order]
    parts = ax.violinplot(data, vert=False, showmedians=True, widths=0.8)
    for pc in parts["bodies"]: pc.set_facecolor(CAMPAIGNS[C_CUDA]["colour"]); pc.set_alpha(0.45)
    for k in ("cbars", "cmins", "cmaxes", "cmedians"): parts[k].set_color("0.3")
    ax.axvline(1, color=C_REF, linestyle=":", linewidth=1.3); ax.set_xscale("log")
    ax.set_yticks(range(1, len(order) + 1), order, fontsize=9)
    ax.set_xlabel("Inter-run CoV per sweep point (%)"); ax.set_title("Spread of the 10 CUDA runs", fontweight="bold")
    ax.grid(True, axis="x", which="both", alpha=0.25)
    med = [DISP[(C_CUDA, b)].cov_pct.median() for b in order]
    p90 = [DISP[(C_CUDA, b)].cov_pct.quantile(0.9) for b in order]; y = np.arange(len(order))
    ax2.barh(y, p90, color="#0072B2", alpha=0.25, height=0.62, label="90th percentile")
    ax2.barh(y, med, color="#0072B2", height=0.62, label="median")
    ax2.set_yticks(y, order, fontsize=9); ax2.set_xscale("log")
    ax2.axvline(1, color=C_REF, linestyle=":", linewidth=1.3)
    ax2.set_xlabel("Inter-run CoV (%)"); ax2.set_title("Median and 90th percentile per benchmark", fontweight="bold")
    ax2.grid(True, axis="x", which="both", alpha=0.25); ax2.legend(fontsize=8, loc="lower right")
    fig.suptitle("Part 1 - reproducibility of the 10 CUDA runs on RTX 2080 Ti", fontsize=12.5, fontweight="bold")
    fig.tight_layout()
    p = FIGDIR / CAMPAIGNS[C_CUDA]["subdir"] / "p1_cuda2080_reproductibilite.png"
    fig.savefig(p, dpi=150, bbox_inches="tight"); plt.show()

def figure_synthese_part2():
    order = sorted(CMP_RES, key=lambda b: CMP_RES[b]["rap"].median())
    fig, ax = plt.subplots(figsize=(12.5, 5.6))
    data = [100 * (CMP_RES[b]["rap"].values - 1) for b in order]
    bp = ax.boxplot(data, vert=False, widths=0.62, patch_artist=True, showfliers=False,
                    medianprops=dict(color="0.15", linewidth=1.6))
    for patch, b in zip(bp["boxes"], order):
        patch.set_facecolor("#D55E00" if CMP_RES[b]["better"] == "min" else "#0072B2"); patch.set_alpha(0.45)
    ax.axvline(0, color="0.2", linestyle="--", linewidth=1.5)
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, p: f"{v:+g} %"))
    ax.set_yticks(range(1, len(order) + 1),
                  [f"{b}  ({'min' if CMP_RES[b]['better']=='min' else 'max'})" for b in order], fontsize=9)
    ax.set_xlabel("HIP deviation from CUDA, on the mean of the 10 runs")
    ax.set_title("Part 2 - HIP relative to CUDA on the same RTX 2080 Ti", fontweight="bold", fontsize=12.5)
    ax.legend(handles=[matplotlib.patches.Patch(color="#0072B2", alpha=0.45, label="maximise: > 0 = HIP faster"),
                       matplotlib.patches.Patch(color="#D55E00", alpha=0.45, label="minimise: > 0 = HIP slower")],
              fontsize=8.5, loc="lower right")
    ax.grid(True, axis="x", which="both", alpha=0.25); fig.tight_layout()
    p = FIGDIR / CAMPAIGNS[C_HIP]["subdir"] / "p2_cuda_vs_hip_synthese.png"
    fig.savefig(p, dpi=150, bbox_inches="tight"); plt.show()

def figure_repro_part3():
    communs = [b for b in BENCHES if (C_CUDA, b) in DISP and (C_MI, b) in DISP]
    x = np.arange(len(communs)); w = 0.38
    fig, ax = plt.subplots(figsize=(13, 5.4))
    ax.bar(x - w/2, [DISP[(C_CUDA, b)].cov_pct.median() for b in communs], w,
           color=CAMPAIGNS[C_CUDA]["colour"], label="CUDA / 2080 Ti (part 1)")
    ax.bar(x + w/2, [DISP[(C_MI, b)].cov_pct.median() for b in communs], w,
           color=CAMPAIGNS[C_MI]["colour"], label="HIP / MI210 (part 3)")
    for i, b in enumerate(communs):
        ax.plot([i - w/2, i - w/2], [DISP[(C_CUDA, b)].cov_pct.median(), DISP[(C_CUDA, b)].cov_pct.quantile(.9)], color="0.3", linewidth=1.1)
        ax.plot([i + w/2, i + w/2], [DISP[(C_MI, b)].cov_pct.median(), DISP[(C_MI, b)].cov_pct.quantile(.9)], color="0.3", linewidth=1.1)
    ax.axhline(1, color=C_REF, linestyle=":", linewidth=1.3); ax.set_yscale("log")
    ax.set_xticks(x, communs, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Median inter-run CoV (%)")
    ax.set_title("Inter-run spread, CUDA / 2080 Ti versus HIP / MI210", fontweight="bold", fontsize=12.5)
    ax.grid(True, axis="y", which="both", alpha=0.25); ax.legend(fontsize=9); fig.tight_layout()
    p = FIGDIR / CAMPAIGNS[C_MI]["subdir"] / "p3_mi210_reproductibilite.png"
    fig.savefig(p, dpi=150, bbox_inches="tight"); plt.show()

## Part 1 - native CUDA on the RTX 2080 Ti

In [ ]:
figure_benchmark(C_CUDA, "gpu-latency")

In [ ]:
figure_benchmark(C_CUDA, "gpu-cache")

In [ ]:
figure_benchmark(C_CUDA, "gpu-l2-stream")

In [ ]:
figure_benchmark(C_CUDA, "gpu-strides")

In [ ]:
figure_benchmark(C_CUDA, "gpu-small-kernels")

In [ ]:
figure_benchmark(C_CUDA, "gpu-roofline")

In [ ]:
figure_benchmark(C_CUDA, "gpu-memcpy")

In [ ]:
figure_benchmark(C_CUDA, "gpu-umstream")

In [ ]:
figure_benchmark(C_CUDA, "gpu-incore")

In [ ]:
figure_repro_part1()

## Part 2 - CUDA versus HIP on the same NVIDIA card

In [ ]:
figure_benchmark_compare("gpu-latency")

In [ ]:
figure_benchmark_compare("gpu-cache")

In [ ]:
figure_benchmark_compare("gpu-l2-stream")

In [ ]:
figure_benchmark_compare("gpu-strides")

In [ ]:
figure_benchmark_compare("gpu-small-kernels")

In [ ]:
figure_benchmark_compare("gpu-roofline")

In [ ]:
figure_benchmark_compare("gpu-memcpy")

In [ ]:
figure_benchmark_compare("gpu-umstream")

In [ ]:
figure_benchmark_compare("gpu-incore")

In [ ]:
figure_synthese_part2()

## Part 3 - HIP on the AMD MI210

In [ ]:
figure_benchmark(C_MI, "gpu-latency")

In [ ]:
figure_benchmark(C_MI, "gpu-cache")

In [ ]:
figure_benchmark(C_MI, "gpu-l2-stream")

In [ ]:
figure_benchmark(C_MI, "gpu-strides")

In [ ]:
figure_benchmark(C_MI, "gpu-small-kernels")

In [ ]:
figure_benchmark(C_MI, "gpu-roofline")

In [ ]:
figure_benchmark(C_MI, "gpu-memcpy")

In [ ]:
figure_benchmark(C_MI, "gpu-umstream")

In [ ]:
figure_benchmark(C_MI, "gpu-incore")

In [ ]:
figure_repro_part3()